# INGESTAO DOS DADOS

In [1]:
# =====================================================
# 1. IMPORTS E CONFIGURAÇÃO
# =====================================================
# Importa bibliotecas necessárias para o pipeline

import os
import requests
import gzip
import shutil
import tarfile
import pandas as pd
import glob
import gc

In [2]:
# =====================================================
# 2. CRIAR ESTRUTURA DE PASTAS
# =====================================================
# Organiza o projeto em camadas de dados

os.makedirs("ifood_case/data/raw", exist_ok=True)
os.makedirs("ifood_case/data/processed", exist_ok=True)
os.makedirs("ifood_case/data/curated", exist_ok=True)

In [3]:
# =====================================================
# 3. DEFINIR URLs DOS DATASETS
# =====================================================

url_orders = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/order.json.gz"

url_consumers = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/consumer.csv.gz"

url_restaurants = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/restaurant.csv.gz"

url_ab_test = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/ab_test_ref.tar.gz"

In [4]:
# =====================================================
# 4. FUNÇÃO DE DOWNLOAD (STREAM PARA ARQUIVOS GRANDES)
# =====================================================
# Baixa arquivos grandes sem estourar memória

def download_file(url, path):

    response = requests.get(url, stream=True)

    response.raise_for_status()

    with open(path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024*1024):
            if chunk:
                f.write(chunk)

    print(f"Download concluído: {path}")

In [5]:
# =====================================================
# 5. DOWNLOAD DOS DATASETS
# =====================================================

download_file(url_orders, "ifood_case/data/raw/order.json.gz")

download_file(url_consumers, "ifood_case/data/raw/consumer.csv.gz")

download_file(url_restaurants, "ifood_case/data/raw/restaurant.csv.gz")

download_file(url_ab_test, "ifood_case/data/raw/ab_test_ref.tar.gz")

Download concluído: ifood_case/data/raw/order.json.gz
Download concluído: ifood_case/data/raw/consumer.csv.gz
Download concluído: ifood_case/data/raw/restaurant.csv.gz
Download concluído: ifood_case/data/raw/ab_test_ref.tar.gz


In [6]:
# =====================================================
# 6. DESCOMPACTAR ORDERS
# =====================================================

with gzip.open("ifood_case/data/raw/order.json.gz", "rb") as f_in:

    with open("ifood_case/data/raw/order.json", "wb") as f_out:

        shutil.copyfileobj(f_in, f_out)

print("Arquivo order.json descompactado")

Arquivo order.json descompactado


In [7]:
# =====================================================
# 7. PROCESSAR ORDERS EM CHUNKS
# =====================================================
# O JSON é grande, então processamos em pedaços

chunks = pd.read_json(
    "ifood_case/data/raw/order.json",
    lines=True,
    chunksize=50000
)

for i, chunk in enumerate(chunks):

    chunk.to_parquet(
        f"ifood_case/data/processed/orders_part_{i}.parquet",
        index=False
    )

    #print(f"Chunk salvo: {i}")

    del chunk
    gc.collect()

print("Orders processado com sucesso")

Orders processado com sucesso


In [8]:
# =====================================================
# 8. PROCESSAR CONSUMERS
# =====================================================

chunks = pd.read_csv(
    "ifood_case/data/raw/consumer.csv.gz",
    compression="gzip",
    chunksize=50000
)

for i, chunk in enumerate(chunks):

    chunk.to_parquet(
        f"ifood_case/data/processed/consumers_part_{i}.parquet",
        index=False
    )

    #print(f"Chunk salvo: {i}")

    del chunk
    gc.collect()

print("Consumers processado")

Consumers processado


In [9]:
# =====================================================
# 9. PROCESSAR RESTAURANTS
# =====================================================

chunks = pd.read_csv(
    "ifood_case/data/raw/restaurant.csv.gz",
    compression="gzip",
    chunksize=50000
)

for i, chunk in enumerate(chunks):

    chunk.to_parquet(
        f"ifood_case/data/processed/restaurants_part_{i}.parquet",
        index=False
    )

    #print(f"Chunk salvo: {i}")

    del chunk
    gc.collect()

print("Restaurants processado")

Restaurants processado


In [10]:
# =====================================================
# 11. EXTRAIR E CARREGAR AB TEST
# =====================================================
import pandas as pd
import tarfile
import os
import gc

raw_tar_path = "ifood_case/data/raw/ab_test_ref.tar.gz"
processed_folder = "ifood_case/data/processed/"

print("Extraindo o arquivo correto do pacote tar.gz...")
with tarfile.open(raw_tar_path, "r:gz") as tar:
    # A grande mudança: ignorar arquivos que começam com '._'
    files_inside = [m for m in tar.getmembers() if m.isfile() and not m.name.startswith("._")]

    # Agora o [0] vai pegar o arquivo correto de 58MB
    target_file = files_inside[0]

    tar.extract(target_file, path="ifood_case/data/raw/")
    extracted_file_path = os.path.join("ifood_case/data/raw/", target_file.name)

print(f"Arquivo '{target_file.name}' extraído! Iniciando fatiamento...")

# Processando o arquivo correto com o encoding para evitar erros de caracteres
chunks = pd.read_csv(extracted_file_path, chunksize=50000, encoding='latin1')

for i, chunk in enumerate(chunks):
    chunk.to_parquet(
        f"{processed_folder}ab_test_part_{i}.parquet",
        index=False
    )
    #print(f"Chunk salvo: {i}")

    del chunk
    gc.collect()

print("Processamento da tabela de Teste A/B concluído com sucesso!")

Extraindo o arquivo correto do pacote tar.gz...


/tmp/ipykernel_35034/2776162763.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(target_file, path="ifood_case/data/raw/")


Arquivo 'ab_test_ref.csv' extraído! Iniciando fatiamento...
Processamento da tabela de Teste A/B concluído com sucesso!


In [11]:
# =====================================================
# 12. CARREGAR ORDERS
# =====================================================

import glob
files = glob.glob("ifood_case/data/processed/orders_part_*.parquet")

orders = pd.concat(
    [pd.read_parquet(f) for f in files]
)

In [12]:
# =====================================================
# 13. CARREGAR CONSUMERS
# =====================================================

import glob
files = glob.glob("ifood_case/data/processed/consumers_part_*.parquet")

consumers = pd.concat(
    [pd.read_parquet(f) for f in files]
)

In [13]:
# =====================================================
# 14. CARREGAR RESTAURANTS
# =====================================================
import glob
files = glob.glob("ifood_case/data/processed/restaurants_part_*.parquet")

restaurants = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

print("Restaurants:", restaurants.shape)

Restaurants: (7292, 12)


In [14]:
restaurants.shape

(7292, 12)

In [15]:
# =====================================================
# 15. CARREGAR TESTE AB
# =====================================================
import tarfile

raw_tar_path = "ifood_case/data/raw/ab_test_ref.tar.gz"

print("Inspecionando o conteúdo do pacote:\n")

with tarfile.open(raw_tar_path, "r:gz") as tar:
    for member in tar.getmembers():
        print(f"Nome: {member.name} | É arquivo? {member.isfile()} | Tamanho: {member.size} bytes")


Inspecionando o conteúdo do pacote:

Nome: ._ab_test_ref.csv | É arquivo? True | Tamanho: 176 bytes
Nome: ab_test_ref.csv | É arquivo? True | Tamanho: 58426128 bytes


In [16]:
# =====================================================
# 16. ENRIQUECIMENTO DA TABELA DE PEDIDOS (FACT) COM DIMENSÕES EM PROCESSAMENTO POR LOTES
# =====================================================
import pandas as pd
import glob
import gc
import os

# 1. Cria a pasta final para os dados cruzados
os.makedirs("ifood_case/data/analytics", exist_ok=True)

print("1. Carregando e LIMPANDO as tabelas menores (Dimensões)...")

# Carrega e REMOVE DUPLICATAS para evitar a "explosão de memória" no join
df_ab_test = pd.concat([pd.read_parquet(f) for f in glob.glob("ifood_case/data/processed/ab_test_part_*.parquet")], ignore_index=True)
df_ab_test = df_ab_test.drop_duplicates(subset=['customer_id']) # Garante ID único

df_consumers = pd.concat([pd.read_parquet(f) for f in glob.glob("ifood_case/data/processed/consumers_part_*.parquet")], ignore_index=True)
df_consumers = df_consumers.drop_duplicates(subset=['customer_id']) # Garante ID único

df_restaurants = pd.concat([pd.read_parquet(f) for f in glob.glob("ifood_case/data/processed/restaurants_part_*.parquet")], ignore_index=True)
df_restaurants = df_restaurants.drop_duplicates(subset=['id']) # Garante ID único

print("Tabelas higienizadas carregadas! RAM sob controle.")
print("2. Iniciando o cruzamento com a tabela de Pedidos (Fato) em lotes...")

# Mapeia todos os pedaços da tabela gigante de pedidos
arquivos_orders = glob.glob("ifood_case/data/processed/orders_part_*.parquet")

for i, arquivo in enumerate(arquivos_orders):
    # Lê APENAS um pedaço dos pedidos por vez
    df_chunk = pd.read_parquet(arquivo)

    # Faz os left joins (agora blindados contra explosão)
    df_chunk = df_chunk.merge(df_ab_test, on="customer_id", how="left")

    df_chunk = df_chunk.merge(df_consumers, on="customer_id", how="left", suffixes=('', '_consumer'))

    df_chunk = df_chunk.merge(
        df_restaurants,
        left_on="merchant_id",
        right_on="id",
        how="left",
        suffixes=('', '_restaurant')
    )

    # Salva o lote já enriquecido na pasta final
    df_chunk.to_parquet(
        f"ifood_case/data/analytics/tabelao_final_part_{i}.parquet",
        #f"/content/drive/MyDrive/ifood_case/data/analytics/tabelao_final_part_{i}.parquet", #google drive - nao considerar

        index=False
    )
    #print(f"Lote {i} cruzado e salvo com sucesso!")

    # Faxina rigorosa na RAM
    del df_chunk
    gc.collect()

print("\n Todos os dados foram cruzados com sucesso!")

1. Carregando e LIMPANDO as tabelas menores (Dimensões)...
Tabelas higienizadas carregadas! RAM sob controle.
2. Iniciando o cruzamento com a tabela de Pedidos (Fato) em lotes...

 Todos os dados foram cruzados com sucesso!


In [17]:
# =====================================================
# 17. PREVIEW DA TABELA
# =====================================================

# 1. Pega a lista de arquivos do Tabelão
arquivos_tabelao = glob.glob("ifood_case/data/analytics/tabelao_final_part_*.parquet")
#arquivos_tabelao = glob.glob("/content/drive/MyDrive/ifood_case/data/analytics/tabelao_final_part_*.parquet") #google drive - nao considerar


if len(arquivos_tabelao) > 0:
    # 2. Lê apenas o PRIMEIRO arquivo da lista (índice 0)
    df_amostra = pd.read_parquet(arquivos_tabelao[0])

    print(f"Mostrando as primeiras 15 linhas do Tabelão (Total de colunas: {len(df_amostra.columns)}):")

    # 3. Exibe as 5 primeiras linhas
    display(df_amostra.head(5))
    print(df_amostra.columns.tolist())


Mostrando as primeiras 15 linhas do Tabelão (Total de colunas: 41):


,cpf,customer_id,customer_name,delivery_address_city,delivery_address_country,delivery_address_district,delivery_address_external_id,delivery_address_latitude,delivery_address_longitude,delivery_address_state,...,enabled,price_range,average_ticket,takeout_time,delivery_time,minimum_order_value,merchant_zip_code,merchant_city,merchant_state,merchant_country
0,2300770190,a36f697a7011b1b5ba1c6304a14b7d221eb9c4aa7fb995...,VICTOR,SAO CAETANO DO SUL,BR,CERAMICA,8015750,-46.57,-23.63,SP,...,True,1,30.0,20,0.0,12.0,92200,SANTO ANDRE,SP,BR
1,71529196630,423b122758205da373abdbb176160864e0ae3dc364eb63...,SILVIA,SAO PAULO,BR,VILA CANERO,8657938,-46.57,-23.57,SP,...,False,3,60.0,0,40.0,20.0,31572,SAO PAULO,SP,BR
2,15180012441,8f91d8ee00d19b67641bd3250e61b638233c1d7c529502...,CEZÁRIO,MARILIA,BR,FRAGATA,7746188,-49.94,-22.22,SP,...,True,1,30.0,0,0.0,0.0,17502,MARILIA,SP,BR
3,13012256457,857ece728c320fc33802ef8c48ab034f666129b0cecfdb...,GUILHERME,SAO PAULO,BR,VILA PRUDENTE,3333670,-46.58,-23.58,SP,...,False,3,80.0,0,55.0,40.0,31310,SAO PAULO,SP,BR
4,38495996372,7d5655de1315a40b7fa6847b6cdc882481e46f3a6c2e0f...,CAROLINA,RIO DE JANEIRO,BR,TIJUCA,2251217,-43.25,-22.93,RJ,...,True,4,60.0,0,30.0,0.0,20550,RIO DE JANEIRO,RJ,BR


['cpf', 'customer_id', 'customer_name', 'delivery_address_city', 'delivery_address_country', 'delivery_address_district', 'delivery_address_external_id', 'delivery_address_latitude', 'delivery_address_longitude', 'delivery_address_state', 'delivery_address_zip_code', 'items', 'merchant_id', 'merchant_latitude', 'merchant_longitude', 'merchant_timezone', 'order_created_at', 'order_id', 'order_scheduled', 'order_total_amount', 'origin_platform', 'order_scheduled_date', 'is_target', 'language', 'created_at', 'active', 'customer_name_consumer', 'customer_phone_area', 'customer_phone_number', 'id', 'created_at_restaurant', 'enabled', 'price_range', 'average_ticket', 'takeout_time', 'delivery_time', 'minimum_order_value', 'merchant_zip_code', 'merchant_city', 'merchant_state', 'merchant_country']


In [18]:
len(arquivos_tabelao)

74

# =====================================================
# 1. CARREGAR TABELA ANALÍTICA
# =====================================================

arquivos = glob.glob("ifood_case/data/analytics/tabelao_final_part_*.parquet")

df = pd.concat([pd.read_parquet(f) for f in arquivos], ignore_index=True)

print("Dataset carregado:", df.shape)

In [19]:
# =====================================================
# 1. CARREGAR TABELA ANALÍTICA (SEM ESTOURAR RAM) => APAGAR DEPOIS
# =====================================================

import pandas as pd
import glob

arquivos = glob.glob("ifood_case/data/analytics/tabelao_final_part_*.parquet")
#arquivos = glob.glob("/content/drive/MyDrive/ifood_case/data/analytics/tabelao_final_part_*.parquet")

total_linhas = 0

for f in arquivos:

    df_chunk = pd.read_parquet(f)

    total_linhas += len(df_chunk)

    del df_chunk

print("Total de linhas no dataset:", total_linhas)

Total de linhas no dataset: 3670826


# ANALISE DE DADOS

1. No iFood, várias áreas utilizam testes A/B para avaliar o impacto de ações em diferentes
métricas. Esses testes permitem validar hipóteses de crescimento e a viabilidade de novas
funcionalidades em um grupo restrito de usuários. Nos dados fornecidos nesse case você
encontrará uma marcação de usuários, separando-os entre grupo teste e controle de uma
campanha de cupons, que disponibilizou para os usuários do grupo teste um cupom especial.
a) Defina os indicadores relevantes para mensurar o sucesso da campanha e analise se ela
teve impacto significativo dentro do período avaliado.
b) Faça uma análise de viabilidade financeira dessa iniciativa como alavanca de
crescimento, adotando as premissas que julgar necessárias (explicite as premissas
adotadas).
c) Recomende oportunidades de melhoria nessa ação e desenhe uma nova proposta de
teste A/B para validar essas hipóteses.

MÉTRICA 1 — VOLUME DE PEDIDOS

Definição
Comparar quantidade de pedidos entre:

controle (is_target = 0)

teste (is_target = 1)

Se teste > controle, o cupom aumentou pedidos.

In [20]:
# =====================================================
# MÉTRICA 1 — VOLUME DE PEDIDOS
# =====================================================

import pandas as pd
import glob

arquivos = glob.glob("ifood_case/data/analytics/tabelao_final_part_*.parquet")
#arquivos = glob.glob("/content/drive/MyDrive/ifood_case/data/analytics/tabelao_final_part_*.parquet")

pedidos_controle = 0
pedidos_teste = 0

for f in arquivos:

    df_chunk = pd.read_parquet(f)

    pedidos_controle += df_chunk[df_chunk["is_target"] == "control"]["order_id"].count()
    pedidos_teste += df_chunk[df_chunk["is_target"] == "target"]["order_id"].count()

    del df_chunk

print("Pedidos controle - control:", pedidos_controle)
print("Pedidos teste - target:", pedidos_teste)

Pedidos controle - control: 1525576
Pedidos teste - target: 2145250


**Interpretação:**

O grupo que recebeu cupom (target) realizou mais pedidos.
Isso indica que a campanha aumentou a frequência de compra.

**Conclusão:**

O grupo exposto ao cupom apresentou maior volume de pedidos em relação ao grupo controle, indicando que a campanha incentivou usuários a realizarem mais compras durante o período analisado.

**MÉTRICA 2 — RECEITA TOTAL (GMV)**

Definição

Verificar se o grupo com cupom gerou mais receita total.

In [21]:
# =====================================================
# MÉTRICA 2 — RECEITA TOTAL (GMV)
# =====================================================

import pandas as pd
import glob

arquivos = glob.glob("ifood_case/data/analytics/tabelao_final_part_*.parquet")
#arquivos = glob.glob("/content/drive/MyDrive/ifood_case/data/analytics/tabelao_final_part_*.parquet")

receita_controle = 0
receita_teste = 0

for f in arquivos:

    df_chunk = pd.read_parquet(f)

    receita_controle += df_chunk[df_chunk["is_target"] == "control"]["order_total_amount"].sum()
    receita_teste += df_chunk[df_chunk["is_target"] == "target"]["order_total_amount"].sum()

    del df_chunk

print("Receita controle:", receita_controle)
print("Receita teste:", receita_teste)

Receita controle: 73071872.87999997
Receita teste: 102760894.97000003



**Resultados:**

Receita controle: 73.071.872,88
Receita teste: 102.760.894,97

**Diferença:**

+29.689.022,09 de receita no grupo teste

**Interpretação:**

O grupo target (com cupom) gerou mais receita total.

Isso indica que a campanha não apenas aumentou pedidos, mas também a receita do período.


**MÉTRICA 3 — TICKET MÉDIO**

Definição

Verificar se o cupom fez as pessoas:

gastar mais

gastar menos

gastar igual

Fórmula:

Ticket médio = Receita total / Número de pedidos

In [22]:
# =====================================================
# MÉTRICA 3 — TICKET MÉDIO
# =====================================================

ticket_controle = receita_controle / pedidos_controle
ticket_teste = receita_teste / pedidos_teste

print("Ticket médio controle:", ticket_controle)
print("Ticket médio teste:", ticket_teste)

Ticket médio controle: 47.89789094741918
Ticket médio teste: 47.901594205803534


**Resultados:**

Ticket médio controle: 47.90
Ticket médio teste: 47.90

**Interpretação:**

O ticket médio é praticamente igual entre os grupos.

O cupom não aumentou nem reduziu o valor gasto por pedido.

**MÉTRICA 4 — USUÁRIOS COMPRADORES**

Definição

Medir quantos clientes únicos fizeram pedidos.

Isso mostra se o cupom:

trouxe mais clientes ativos

ou apenas fez os mesmos clientes comprarem mais

In [23]:
# =====================================================
# MÉTRICA 4 — USUÁRIOS COMPRADORES
# =====================================================

import pandas as pd
import glob

arquivos = glob.glob("ifood_case/data/analytics/tabelao_final_part_*.parquet")
#arquivos = glob.glob("/content/drive/MyDrive/ifood_case/data/analytics/tabelao_final_part_*.parquet")

usuarios_controle = set()
usuarios_teste = set()

for f in arquivos:

    df_chunk = pd.read_parquet(f)

    usuarios_controle.update(
        df_chunk[df_chunk["is_target"] == "control"]["customer_id"].unique()
    )

    usuarios_teste.update(
        df_chunk[df_chunk["is_target"] == "target"]["customer_id"].unique()
    )

    del df_chunk

print("Usuários controle:", len(usuarios_controle))
print("Usuários teste:", len(usuarios_teste))

Usuários controle: 360542
Usuários teste: 445925


**Resultados:**

Usuários controle: 360.542
Usuários teste: 445.925

**Diferença:**

+85.383 usuários compradores no grupo teste

**Interpretação:**

Mais clientes únicos fizeram pedidos no grupo com cupom.

Isso indica que a campanha aumentou a base de usuários ativos.

**MÉTRICA 5 — PEDIDOS POR USUÁRIO**

Definição

Medir se os clientes compraram com mais frequência.

Fórmula:

Pedidos por usuário = número de pedidos / usuários compradores

In [24]:
# =====================================================
# MÉTRICA 5 — PEDIDOS POR USUÁRIO
# =====================================================

pedidos_por_usuario_controle = pedidos_controle / len(usuarios_controle)
pedidos_por_usuario_teste = pedidos_teste / len(usuarios_teste)

print("Pedidos por usuário controle:", pedidos_por_usuario_controle)
print("Pedidos por usuário teste:", pedidos_por_usuario_teste)

Pedidos por usuário controle: 4.231340592774212
Pedidos por usuário teste: 4.810786567247855


**Resultados:**

Pedidos por usuário controle: 4.23
Pedidos por usuário teste: 4.81

**Diferença: **

+0.58 pedidos por usuário no grupo teste

**Interpretação: **

Usuários que receberam cupom compraram com maior frequência.

O cupom incentivou repetição de compra.

Conclusão Geral da Questão 1a

Resumo dos resultados:

Métrica	Resultado
Pedidos	maior no grupo teste
Receita	maior no grupo teste
Ticket médio	praticamente igual
Usuários compradores	maior no grupo teste
Pedidos por usuário	maior no grupo teste

Conclusão:

A campanha de cupons apresentou impacto positivo no comportamento de compra dos usuários. O grupo teste apresentou maior número de pedidos, maior receita total, mais usuários compradores e maior frequência de compra. O ticket médio permaneceu estável, indicando que o aumento de receita foi impulsionado principalmente pelo maior volume de pedidos.

**1b — Viabilidade Financeira da Campanha**

Definição

Agora precisamos avaliar se o aumento de receita compensa o custo dos cupons.

Como o dataset não informa o valor do cupom, precisamos assumir premissas.

Premissas simples para análise:

cupom médio: R$10

somente grupo target recebeu cupom

assumimos 1 cupom por pedido

Depois podemos testar outros cenários.

Passo 1 — Calcular pedidos adicionais gerados

Fórmula:

Pedidos adicionais = pedidos_teste - pedidos_controle

In [25]:
# =====================================================
# PEDIDOS ADICIONAIS GERADOS PELA CAMPANHA
# =====================================================

pedidos_incrementais = pedidos_teste - pedidos_controle

print("Pedidos incrementais:", pedidos_incrementais)

Pedidos incrementais: 619674


Passo 2 — Receita Incremental

Definição
Receita adicional gerada pela campanha.

Fórmula:

Receita incremental = receita_teste - receita_controle

In [26]:
# =====================================================
# RECEITA INCREMENTAL
# =====================================================

receita_incremental = receita_teste - receita_controle

print("Receita incremental:", receita_incremental)

Receita incremental: 29689022.090000063


Passo 3 — Custo da Campanha

Premissa adotada:

Valor médio do cupom = R$10

Assumindo 1 cupom por pedido no grupo teste.

Fórmula:

Custo campanha = pedidos_teste × valor_cupom

In [27]:
# =====================================================
# CUSTO DA CAMPANHA
# =====================================================

valor_cupom = 10

custo_campanha = pedidos_teste * valor_cupom

print("Custo da campanha:", custo_campanha)

Custo da campanha: 21452500


Passo 4 — Lucro Incremental

Definição
Verificar se a receita adicional cobre o custo dos cupons.

Fórmula:

Lucro incremental = receita_incremental - custo_campanha

In [28]:
# =====================================================
# LUCRO INCREMENTAL DA CAMPANHA
# =====================================================

lucro_incremental = receita_incremental - custo_campanha

print("Lucro incremental:", lucro_incremental)

Lucro incremental: 8236522.090000063


Conclusão — 1b Viabilidade Financeira

Resultados:

Receita incremental: R$ 29.689.022
Custo da campanha: R$ 21.452.500
Lucro incremental: R$ 8.236.522

Interpretação:

A receita gerada pela campanha superou o custo dos cupons.

A campanha gerou aproximadamente R$ 8,2 milhões de ganho incremental.

1c — Oportunidades de melhoria da campanha

Principais aprendizados da análise:

o cupom aumentou pedidos

aumentou usuários compradores

aumentou frequência de compra

não aumentou ticket médio

Isso indica que o cupom estimula volume, mas não aumenta gasto por pedido.

Melhorias sugeridas
1️⃣ Cupom com valor mínimo de pedido

Objetivo: aumentar ticket médio.

Exemplo:

R$10 de desconto em pedidos acima de R$60
2️⃣ Segmentação por usuários menos ativos

Enviar cupons para:

usuários com baixo histórico de pedidos

usuários inativos há mais tempo

Objetivo:

aumentar ativação de clientes
3️⃣ Testar diferentes valores de cupom

Exemplo:

Grupo	Cupom
Controle	sem cupom
Teste A	R$5
Teste B	R$10
Teste C	R$15

Objetivo:

identificar o menor incentivo necessário para gerar conversão
Novo desenho de teste A/B

Proposta de experimento:

Grupos
Grupo	Regra
Controle	sem cupom
Teste 1	cupom R$10
Teste 2	cupom R$10 com pedido mínimo
Métricas de avaliação

Principais:

número de pedidos

receita total

ticket médio

pedidos por usuário

Financeiras:

receita incremental

custo de cupons

lucro incremental

Conclusão para o case

A campanha demonstrou impacto positivo em volume de pedidos e receita, porém não apresentou efeito relevante no ticket médio. Recomenda-se testar novas estratégias de cupons com restrição de valor mínimo de pedido e diferentes valores de incentivo. Um novo teste A/B pode avaliar qual configuração maximiza conversão mantendo viabilidade financeira.

2. A criação de segmentações permite agrupar usuários de acordo com características e
comportamentos similares, possibilitando criar estratégias direcionadas de acordo com o perfil
de cada público, facilitando a personalização e incentivando o engajamento, retenção, além de
otimização de recursos. Segmentações de usuários são muito utilizadas pelos times de Data,
mas a área em que você atua ainda não tem segmentos bem definidos e cada área de Negócio
utiliza conceitos diferentes. Por isso, você precisa:
a) Estabelecer quais serão os critérios utilizados para cada segmento. Utilize os critérios/
ferramentas que achar necessários, mas lembre-se de explicar o racional utilizado na
criação.
b) Analisar os resultados do teste A/B com base nos segmentos definidos e propor ações
específicas para cada um deles

2a — Definição dos Segmentos
Critério utilizado

Segmentação baseada em comportamento de compra.

Variável usada:

Pedidos por usuário

Racional:

identifica nível de engajamento do cliente

muito usado em e-commerce e marketplace

permite criar estratégias diferentes por perfil

Segmentos definidos
Segmento	Critério
Baixa frequência	até 2 pedidos
Média frequência	3 a 5 pedidos
Alta frequência	mais de 5 pedidos

Passo 1 — Calcular pedidos por usuário

In [29]:
# =====================================================
# PEDIDOS POR USUÁRIO
# =====================================================

import pandas as pd
import glob

arquivos = glob.glob("/content/drive/MyDrive/ifood_case/data/analytics/tabelao_final_part_*.parquet")

pedidos_usuario = {}

for f in arquivos:

    df_chunk = pd.read_parquet(f)

    contagem = df_chunk.groupby("customer_id")["order_id"].count()

    for cliente, pedidos in contagem.items():
        pedidos_usuario[cliente] = pedidos_usuario.get(cliente,0) + pedidos

    del df_chunk

print("Total usuários:", len(pedidos_usuario))

Total usuários: 806466


Passo 2 — Criar segmentos

In [30]:
# =====================================================
# CRIAR SEGMENTOS
# =====================================================

segmentos = {
    "baixa":0,
    "media":0,
    "alta":0
}

for pedidos in pedidos_usuario.values():

    if pedidos <= 2:
        segmentos["baixa"] += 1

    elif pedidos <= 5:
        segmentos["media"] += 1

    else:
        segmentos["alta"] += 1

print(segmentos)

{'baixa': 422049, 'media': 191454, 'alta': 192963}


Conclusão — 2a Segmentação

Resultados:

Baixa frequência: 422.049 usuários
Média frequência: 191.454 usuários
Alta frequência: 192.963 usuários

Interpretação:

A maior parte da base está no segmento baixa frequência.

Existe um grupo relevante de clientes altamente engajados.

Conclusão para o case:

A segmentação foi construída com base na frequência de compra dos usuários, utilizando o número total de pedidos realizados. Esse critério permite identificar diferentes níveis de engajamento e orientar estratégias específicas para cada perfil de cliente.

Segmentos definidos:

Segmento	Critério
Baixa frequência	até 2 pedidos
Média frequência	3 a 5 pedidos
Alta frequência	mais de 5 pedidos

2b — Analisar o teste A/B por segmento

Agora vamos medir quantos pedidos vieram de cada segmento.

In [31]:
# =====================================================
# PEDIDOS POR SEGMENTO
# =====================================================

segmentos_pedidos = {
    "baixa":0,
    "media":0,
    "alta":0
}

for cliente, pedidos in pedidos_usuario.items():

    if pedidos <= 2:
        segmentos_pedidos["baixa"] += pedidos

    elif pedidos <= 5:
        segmentos_pedidos["media"] += pedidos

    else:
        segmentos_pedidos["alta"] += pedidos

print(segmentos_pedidos)

{'baixa': 661511, 'media': 733810, 'alta': 2267000}


Conclusão — 2b Análise por Segmento

Resultados:

Baixa frequência: 661.511 pedidos
Média frequência: 733.810 pedidos
Alta frequência: 2.267.000 pedidos

Interpretação:

O segmento alta frequência concentra a maior parte dos pedidos.

Clientes mais engajados geram a maior parte da receita da plataforma.

Ações recomendadas por segmento
Baixa frequência

Perfil:

clientes pouco ativos

Objetivo:

ativação

Ações:

cupons agressivos
campanhas de reativação
notificações push
Média frequência

Perfil:

clientes em fase de crescimento

Objetivo:

aumentar frequência

Ações:

cupons com valor mínimo de pedido
programas de fidelidade
ofertas personalizadas
Alta frequência

Perfil:

clientes altamente engajados

Objetivo:

retenção

Ações:

benefícios exclusivos
cashback
programas VIP
Conclusão para o case

A segmentação baseada na frequência de compra permite identificar diferentes perfis de usuários e direcionar estratégias específicas para cada grupo. Clientes de alta frequência concentram a maior parte dos pedidos e devem ser foco de estratégias de retenção, enquanto usuários de baixa frequência podem ser estimulados por campanhas de ativação e incentivos promocionais.

import pandas as pd
import glob

arquivos = glob.glob("/content/drive/MyDrive/ifood_case/data/analytics/tabelao_final_part_*.parquet")

df = pd.read_parquet(arquivos[0])

print(df["is_target"].unique())
print(df["is_target"].dtype)